# Model Evaluation — Corrected Version

This notebook is aligned with the current project setup:

- Dataset root: `artifacts/data_ingestion/Chicken Disease`
- Classes: `Coccidiosis`, `Healthy`, `Salmonella`
- Validation split: `0.20` (same as training)
- Preprocessing: VGG16 `preprocess_input` (same as the corrected training component)
- Trained model: `artifacts/training/model.h5`
- Evaluation output: `scores.json`


In [1]:
import os
from pathlib import Path


def find_project_root() -> Path:
    """Find the project root containing config/config.yaml and params.yaml."""
    cwd = Path.cwd()
    candidates = [cwd, *cwd.parents, Path(r"F:\Chicken-Disease-Classification")]

    for candidate in candidates:
        if (
            (candidate / "config" / "config.yaml").exists()
            and (candidate / "params.yaml").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing config/config.yaml and params.yaml."
    )


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("config.yaml exists:", Path("config/config.yaml").exists())
print("params.yaml exists:", Path("params.yaml").exists())


Project root: f:\Chicken-Disease-Classification
config.yaml exists: True
params.yaml exists: True


In [2]:
from dataclasses import dataclass
from pathlib import Path

import tensorflow as tf

from cnnClassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.12.0


## 1. Evaluation configuration entity


In [3]:
@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int


## 2. Configuration manager


In [4]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_validation_config(self) -> EvaluationConfig:
        training_data = (
            Path(self.config.data_ingestion.unzip_dir) / "Chicken Disease"
        )

        all_params = (
            self.params.to_dict()
            if hasattr(self.params, "to_dict")
            else dict(self.params)
        )

        return EvaluationConfig(
            path_of_model=Path("artifacts/training/model.h5"),
            training_data=training_data,
            all_params=all_params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )


## 3. Pre-evaluation checks

This verifies that the trained model and all three class folders exist before evaluation starts.


In [5]:
config = ConfigurationManager()
val_config = config.get_validation_config()

print("Model path:", val_config.path_of_model)
print("Dataset path:", val_config.training_data)

if not val_config.path_of_model.exists():
    raise FileNotFoundError(
        f"Trained model not found: {val_config.path_of_model}"
    )

if not val_config.training_data.exists():
    raise FileNotFoundError(
        f"Dataset directory not found: {val_config.training_data}"
    )

class_folders = sorted(
    p.name for p in val_config.training_data.iterdir() if p.is_dir()
)

expected_classes = ["Coccidiosis", "Healthy", "Salmonella"]

print("Detected class folders:", class_folders)

if class_folders != expected_classes:
    raise ValueError(
        f"Expected class folders {expected_classes}, but found {class_folders}"
    )

print("Pre-evaluation checks passed.")


[2026-08-25 16:57:07,216: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-25 16:57:07,220: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-25 16:57:07,220: INFO: common: created directory at: artifacts]
Model path: artifacts\training\model.h5
Dataset path: artifacts\data_ingestion\Chicken Disease
Detected class folders: ['Coccidiosis', 'Healthy', 'Salmonella']
Pre-evaluation checks passed.


## 4. Evaluation component

`validation_split=0.20` and VGG16 `preprocess_input` intentionally match the corrected training component.


In [7]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
        self.model = None
        self.valid_generator = None
        self.score = None

    def _valid_generator(self):
        datagenerator_kwargs = dict(
            preprocessing_function=tf.keras.applications.vgg16.preprocess_input,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=tuple(self.config.params_image_size[:-1]),
            batch_size=self.config.params_batch_size,
            interpolation="bilinear",
            class_mode="categorical",
            seed=42
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        print("Class mapping:", self.valid_generator.class_indices)

        if self.valid_generator.num_classes != 3:
            raise ValueError(
                f"Expected 3 classes, found {self.valid_generator.num_classes}"
            )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()

        self.score = self.model.evaluate(
            self.valid_generator,
            verbose=1
        )

        print(f"Validation loss: {self.score[0]:.6f}")
        print(f"Validation accuracy: {self.score[1]:.6f}")

    def save_score(self):
        if self.score is None:
            raise RuntimeError("Run evaluation() before save_score().")

        scores = {
            "loss": float(self.score[0]),
            "accuracy": float(self.score[1])
        }

        save_json(
            path=Path("scores.json"),
            data=scores
        )

        print("Saved evaluation scores to scores.json")


## 5. Run evaluation


In [8]:
try:
    config = ConfigurationManager()
    val_config = config.get_validation_config()

    evaluation = Evaluation(config=val_config)
    evaluation.evaluation()
    evaluation.save_score()

    print("Model evaluation completed successfully.")

except Exception as e:
    raise e


[2026-08-25 16:57:28,529: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-25 16:57:28,532: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-25 16:57:28,532: INFO: common: created directory at: artifacts]
Found 1286 images belonging to 3 classes.
Class mapping: {'Coccidiosis': 0, 'Healthy': 1, 'Salmonella': 2}
161/161 [==============================] - 172s 1s/step - loss: 1.4880 - accuracy: 0.9487
Validation loss: 1.488033
Validation accuracy: 0.948678
[2026-08-25 17:00:21,057: INFO: common: json file saved at: scores.json]
Saved evaluation scores to scores.json
Model evaluation completed successfully.


## Expected result

After a successful run, the project root should contain `scores.json`, for example:

```json
{
    "loss": 0.42,
    "accuracy": 0.84
}
```

The numbers above are only an example; your actual values will come from the trained model.
